# E006 - Recherche reproductible des paramètres SRPG

Ce notebook exécute réellement les essais sur la RTX. Il ne cherche pas un joli exemple isolé : il classe les paramètres par lecture exacte sur tous les décodeurs et toutes les dégradations, puis confirme les meilleurs sur plusieurs prompts et seeds. Les résultats sont enregistrés après chaque essai et une relance reprend la campagne.

Références : [article DiffQRCoder v3](https://arxiv.org/abs/2409.06355) et [implémentation officielle](https://github.com/jwliao1209/DiffQRCoder).

## 0. Protocole
Le plan commence par isoler l'effet 40/60/80/100 pas observé sur le téléphone. Il teste ensuite les plages publiées pour $\lambda_1$, $\lambda_2$ et ControlNet, sans produit cartésien inutile. Modifier `EXPERIMENT_NAME` après un changement de code ou de protocole.

In [ ]:
EXPERIMENT_NAME = "e006-srpg-search-v1"
SCREEN_LIMIT = None  # None = les 17 essais; mettre 4 pour vérifier d'abord le pipeline
TOP_COUNT = 3
RUN_CONFIRMATION = True
ERROR_CORRECTION = "H"
NEGATIVE_PROMPT = "easynegative, text, watermark, blurry, plain QR code, barcode"
BASE_STEPS = 12
BASE_STRENGTH = 0.90
BASE_GUIDANCE_SCALE = 12.0
BASE_CONTROLNET_SCALE = 1.50
STAGE2_SEED_OFFSET = 2_000_003
SCREEN_CASE = {
    "name": "botanical",
    "payload": "https://example.prooftag.test/t/e006-a",
    "prompt": (
        "elegant botanical packaging illustration, organic leaves and flowers, "
        "premium print design, high detail"
    ),
    "seed": 42,
}
CONFIRMATION_CASES = [
    {
        "name": "geometric",
        "payload": "https://example.prooftag.test/t/e006-b",
        "prompt": "premium geometric mosaic, blue and gold, intricate editorial illustration",
        "seed": 314,
    },
    {
        "name": "engraving",
        "payload": "https://example.prooftag.test/t/e006-c",
        "prompt": "detailed monochrome botanical engraving, luxury label, organic linework",
        "seed": 2026,
    },
    {
        "name": "abstract",
        "payload": "https://example.prooftag.test/t/e006-d",
        "prompt": "colorful abstract paper cut artwork, flowing shapes, premium graphic design",
        "seed": 9001,
    },
]

## 1. Kernel GPU, modèles et stockage reprenable

In [ ]:
import csv
import html
import json
import time
import traceback
from dataclasses import asdict
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, Markdown, display
from PIL import Image

from prooftag_qr.backends import ControlNetBackend
from prooftag_qr.config import Settings
from prooftag_qr.experiments import (
    aggregate_confirmation,
    screening_trials,
    trial_rank_key,
)
from prooftag_qr.qr import generate_qr, module_error_rate
from prooftag_qr.quality import image_change_metrics
from prooftag_qr.schemas import GenerationRequest
from prooftag_qr.srpg import run_srpg_controlnet_img2img
from prooftag_qr.validation import QRValidator

if not torch.cuda.is_available():
    raise RuntimeError("Ce notebook doit tourner dans le pod RTX du serveur.")
run_dir = Path("/data/parameter-search") / EXPERIMENT_NAME
run_dir.mkdir(parents=True, exist_ok=True)
results_path = run_dir / "results.jsonl"
settings = Settings(
    data_dir=Path("/data"),
    model_cache_dir=Path("/cache"),
    default_backend="controlnet",
    controlnet_pipeline_mode="img2img",
    device="cuda",
    srpg_enabled=False,
    guided_rediffusion_enabled=False,
    latent_refinement_enabled=False,
)
backend = ControlNetBackend(settings)
pipeline = backend._load()
validator = QRValidator()
display(
    Markdown(
        f"**GPU :** `{torch.cuda.get_device_name(0)}`  \
**Campagne :** `{run_dir}`"
    )
)

## 2. Génération des cas, validation et journalisation après chaque essai

In [ ]:
def load_rows():
    if not results_path.exists():
        return []
    return [
        json.loads(line) for line in results_path.read_text(encoding="utf-8").splitlines() if line
    ]


def append_row(row):
    with results_path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(row, ensure_ascii=False) + "\n")


def result_key(phase, case_name, trial_name):
    return f"{phase}:{case_name}:{trial_name}"


def generate_case(case):
    case_dir = run_dir / case["name"]
    case_dir.mkdir(parents=True, exist_ok=True)
    blueprint = generate_qr(case["payload"], ERROR_CORRECTION, size=512)
    raw_path = case_dir / "raw.png"
    if raw_path.exists():
        return blueprint, Image.open(raw_path).convert("RGB")
    request = GenerationRequest(
        payload=case["payload"],
        prompt=case["prompt"],
        negative_prompt=NEGATIVE_PROMPT,
        backend="controlnet",
        error_correction=ERROR_CORRECTION,
        seed=case["seed"],
        steps=BASE_STEPS,
        strength=BASE_STRENGTH,
        guidance_scale=BASE_GUIDANCE_SCALE,
        controlnet_scale=BASE_CONTROLNET_SCALE,
        max_attempts=1,
    )
    raw = backend.generate(request, blueprint, case["seed"])
    raw.save(raw_path)
    blueprint.image.save(case_dir / "qr-control.png")
    return blueprint, raw


def record_raw_baseline(case, blueprint, raw):
    key = result_key("baseline", case["name"], "raw")
    existing = {row["key"]: row for row in load_rows()}
    if key in existing:
        return existing[key]
    records = validator.validate(raw, case["payload"])
    exact = sum(record.exact_payload_match for record in records)
    originals = [record for record in records if record.scenario == "original"]
    raw_path = run_dir / case["name"] / "raw.png"
    raw_path.with_suffix(".validations.json").write_text(
        json.dumps([asdict(record) for record in records], indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    row = {
        "key": key,
        "phase": "baseline",
        "case": case["name"],
        "trial": "raw",
        "status": "ok",
        "timestamp": datetime.now(UTC).isoformat(),
        "qr_version": blueprint.version,
        "matrix_modules": int(blueprint.matrix.shape[0]),
        "pass_rate": exact / len(records),
        "strict_all": exact == len(records),
        "passed": exact,
        "validations": len(records),
        "original_pass_rate": sum(r.exact_payload_match for r in originals) / len(originals),
        "module_error_rate": module_error_rate(raw, blueprint),
        "image": str(raw_path),
    }
    append_row(row)
    return row


def execute_trial(phase, case, trial, blueprint, raw):
    key = result_key(phase, case["name"], trial.name)
    existing = {row["key"]: row for row in load_rows()}
    if key in existing:
        print(f"SKIP {key} (déjà terminé)")
        return existing[key]
    started = time.perf_counter()
    try:
        generator = torch.Generator(device=settings.device).manual_seed(
            (case["seed"] + STAGE2_SEED_OFFSET) % (2**32)
        )
        result = run_srpg_controlnet_img2img(
            pipeline,
            raw,
            blueprint,
            prompt=case["prompt"],
            negative_prompt=NEGATIVE_PROMPT,
            guidance_scale=trial.guidance_scale,
            generator=generator,
            config=trial.to_srpg_config(),
        )
        duration = time.perf_counter() - started
        records = validator.validate(result.image, case["payload"])
        exact = sum(record.exact_payload_match for record in records)
        originals = [record for record in records if record.scenario == "original"]
        original_exact = sum(record.exact_payload_match for record in originals)
        change = image_change_metrics(result.image, raw)
        image_path = run_dir / case["name"] / f"{phase}-{trial.name}.png"
        result.image.save(image_path)
        image_path.with_suffix(".validations.json").write_text(
            json.dumps([asdict(record) for record in records], indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        with image_path.with_suffix(".steps.csv").open("w", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(asdict(result.steps[0]).keys()))
            writer.writeheader()
            writer.writerows(asdict(step) for step in result.steps)
        row = {
            "key": key,
            "phase": phase,
            "case": case["name"],
            "trial": trial.name,
            "status": "ok",
            "timestamp": datetime.now(UTC).isoformat(),
            "parameters": asdict(trial),
            "qr_version": blueprint.version,
            "matrix_modules": int(blueprint.matrix.shape[0]),
            "pass_rate": exact / len(records),
            "strict_all": exact == len(records),
            "passed": exact,
            "validations": len(records),
            "original_pass_rate": original_exact / len(originals),
            "module_error_rate": module_error_rate(result.image, blueprint),
            "mean_absolute_change": change["mean_absolute_change"],
            "changed_pixel_ratio": change["changed_pixel_ratio"],
            "duration_seconds": duration,
            "peak_gpu_memory_mib": result.peak_gpu_memory_allocated_mib,
            "gradient_clip_rate": sum(step.gradient_clipped for step in result.steps)
            / len(result.steps),
            "final_gradient_rms": result.steps[-1].gradient_rms,
            "final_noise_delta_rms": result.steps[-1].noise_delta_rms,
            "internal_accepted": result.accepted,
            "internal_rejection_reason": result.rejection_reason,
            "image": str(image_path),
        }
    except Exception as exc:
        duration = time.perf_counter() - started
        row = {
            "key": key,
            "phase": phase,
            "case": case["name"],
            "trial": trial.name,
            "status": "error",
            "parameters": asdict(trial),
            "duration_seconds": duration,
            "error": repr(exc),
            "traceback": traceback.format_exc(),
        }
        torch.cuda.empty_cache()
    append_row(row)
    print(f"{key}: {row.get('passed', 0)}/{row.get('validations', 0)} en {duration:.1f}s")
    return row


def show_table(rows, columns):
    head = "".join(f"<th>{html.escape(column)}</th>" for column in columns)
    body = []
    for row in rows:
        cells = []
        for column in columns:
            value = row.get(column, "")
            if isinstance(value, float):
                value = f"{value:.4f}"
            cells.append(f"<td>{html.escape(str(value))}</td>")
        body.append("<tr>" + "".join(cells) + "</tr>")
    display(
        HTML(
            "<table><thead><tr>"
            + head
            + "</tr></thead><tbody>"
            + "".join(body)
            + "</tbody></table>"
        )
    )

## 3. Criblage causal sur une image brute fixe
Toutes les variantes utilisent exactement le même brut et la même seed Stage-2. Ainsi, une différence vient des paramètres testés et non d'une nouvelle image aléatoire.

In [ ]:
trials = list(screening_trials())
if SCREEN_LIMIT is not None:
    trials = trials[:SCREEN_LIMIT]
blueprint, raw = generate_case(SCREEN_CASE)
raw_baseline = record_raw_baseline(SCREEN_CASE, blueprint, raw)
display(
    Markdown(
        f"QR version **{blueprint.version}**, matrice **{blueprint.matrix.shape[0]} modules**, "
        f"{len(trials)} essais planifiés."
    )
)
display(raw)
screen_rows = [execute_trial("screen", SCREEN_CASE, trial, blueprint, raw) for trial in trials]

## 4. Classement : lecture stricte d'abord, rendu ensuite
Une image à 100 % des validations passe devant toute image plus jolie mais moins robuste. À taux égal, le classement minimise l'erreur module, la modification du brut puis le temps.

In [ ]:
screen_rows = [
    row
    for row in load_rows()
    if row.get("phase") == "screen" and row.get("case") == SCREEN_CASE["name"]
]
ranked = sorted(screen_rows, key=trial_rank_key)
columns = [
    "trial",
    "status",
    "strict_all",
    "pass_rate",
    "original_pass_rate",
    "module_error_rate",
    "mean_absolute_change",
    "gradient_clip_rate",
    "duration_seconds",
    "peak_gpu_memory_mib",
]
show_table(ranked, columns)
top_rows = [row for row in ranked if row.get("status") == "ok"][:TOP_COUNT]
top_names = [row["trial"] for row in top_rows]
trial_by_name = {trial.name: trial for trial in screening_trials()}
fig, axes = plt.subplots(1, len(top_rows) + 1, figsize=(5 * (len(top_rows) + 1), 5), squeeze=False)
axes = axes.ravel()
axes[0].imshow(raw)
axes[0].set_title("Brut fixe")
axes[0].axis("off")
for axis, row in zip(axes[1:], top_rows, strict=True):
    axis.imshow(Image.open(row["image"]).convert("RGB"))
    axis.set_title(f"{row['trial']}\nscan={row['pass_rate']:.1%}")
    axis.axis("off")
plt.tight_layout()
fig.savefig(run_dir / "screen-top.png", dpi=140)

## 5. Confirmation des trois meilleurs sur d'autres styles et seeds
Le meilleur réglage n'est promu que sur son pire cas. Une moyenne élevée ne masque donc pas un prompt systématiquement illisible.

In [ ]:
if RUN_CONFIRMATION and top_names:
    for case in CONFIRMATION_CASES:
        case_blueprint, case_raw = generate_case(case)
        record_raw_baseline(case, case_blueprint, case_raw)
        for trial_name in top_names:
            execute_trial(
                "confirm",
                case,
                trial_by_name[trial_name],
                case_blueprint,
                case_raw,
            )
confirmation_rows = [
    row for row in load_rows() if row.get("phase") == "confirm" and row.get("trial") in top_names
]
aggregates = aggregate_confirmation(confirmation_rows)
show_table(
    aggregates,
    [
        "trial",
        "cases",
        "all_strict",
        "worst_pass_rate",
        "mean_pass_rate",
        "mean_module_error_rate",
        "mean_absolute_change",
        "mean_duration_seconds",
    ],
)

## 6. Porte physique téléphone et impression
Les décodeurs logiciels ne remplacent pas les téléphones. Le CSV suivant impose dix lectures par condition. Remplir `successes` après les tests ; seul un candidat à 100 % automatique **et** physique pourra devenir le profil de livraison.

In [ ]:
phone_path = run_dir / "phone-validation.csv"
if not phone_path.exists():
    protocols = ["screen-front-30cm", "screen-angle-30deg", "screen-low-light", "print-5cm"]
    with phone_path.open("w", newline="", encoding="utf-8") as stream:
        fields = [
            "trial",
            "case",
            "image",
            "device",
            "protocol",
            "attempts",
            "successes",
            "exact_payload",
            "notes",
        ]
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        physical_candidates = confirmation_rows or top_rows
        for row in physical_candidates:
            for protocol in protocols:
                writer.writerow(
                    {
                        "trial": row["trial"],
                        "case": row["case"],
                        "image": row["image"],
                        "device": "",
                        "protocol": protocol,
                        "attempts": 10,
                        "successes": "",
                        "exact_payload": "",
                        "notes": "",
                    }
                )
display(
    Markdown(
        f"Campagne enregistrée dans **`results/parameter-search/{EXPERIMENT_NAME}`**.  \
"
        f"Compléter **`{phone_path}`** avant toute promotion."
    )
)